# 01 — The Map of Machine Learning: A Taxonomy

**Difficulty:** ⭐⭐ (Beginner–Intermediate)  
**Estimated time:** 2 hours  
**Week:** 5 — ML Fundamentals & Linear Regression

---

## Why does this matter for ML?

Before writing a single line of code, a practitioner must answer: *"What kind of problem am I solving?"*  
Choosing the wrong ML paradigm is like bringing a hammer to a surgery — the tool exists, but it is the wrong one for the job.

This notebook gives you a **mental map** of the ML landscape so you can look at any dataset or business problem and immediately know which family of algorithms to reach for. We will use **California house prices** as our running example throughout — a single domain that stretches across every paradigm.

---

## The Real-World Analogy: Different Ways to Learn

Think about how humans learn different skills:

| Learning Style | Human Analogy | ML Paradigm |
|---|---|---|
| A teacher grades every assignment | You know the right answer for every example | **Supervised** |
| You sort your own bookshelf by noticing patterns | No one tells you which books belong together | **Unsupervised** |
| Teacher grades 10% of homework, you infer the rest | A little labeled data, a lot of unlabeled | **Semi-Supervised** |
| You learn grammar by predicting the next word | Labels are hidden inside the data itself | **Self-Supervised** |
| Learning chess by playing thousands of games | Win/lose signals guide behavior over time | **Reinforcement** |

---

## Notebook Outline

1. Setup & Data Loading  
2. Supervised Learning — Regression  
3. Supervised Learning — Classification  
4. Unsupervised Learning — Clustering  
5. Unsupervised Learning — Dimensionality Reduction  
6. Unsupervised Learning — Anomaly Detection  
7. Semi-Supervised & Self-Supervised (conceptual + code sketch)  
8. Reinforcement Learning (conceptual)  
9. Visual Taxonomy Diagram  
10. Comparison Table  
11. Decision Flowchart  
12. Self-Check Questions

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
# Standard scientific Python stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import seaborn as sns

# sklearn datasets and algorithms we will demonstrate
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, classification_report
)

# Consistent look for all plots
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)          # reproducibility

print('All imports successful!')

In [ ]:
# ── Cell 2: Load Data ────────────────────────────────────────────────────────
# fetch_california_housing returns a Bunch object (like a dict)
housing = fetch_california_housing(as_frame=True)

# The feature matrix X — one row per census block group, one column per feature
X = housing.data

# The target y — median house VALUE in units of $100,000
# e.g. y=2.0 means median price ≈ $200,000
y = housing.target

# Combine into one DataFrame for easy exploration
df = pd.concat([X, y.rename('MedHouseVal')], axis=1)

print(f'Dataset shape: {df.shape}  ({df.shape[0]:,} rows × {df.shape[1]} columns)')
print(f'\nFeatures: {list(X.columns)}')
print(f'Target  : MedHouseVal (median value in $100k)')
print('\nFirst 5 rows:')
df.head()

In [ ]:
# ── Cell 3: Quick Data Summary ───────────────────────────────────────────────
print('=== Basic Statistics ===')
print(df.describe().round(2))

print(f'\nTarget range: ${y.min()*100:.0f}k – ${y.max()*100:.0f}k')
print(f'Median price: ${y.median()*100:.0f}k')
print(f'Missing values: {df.isnull().sum().sum()} (none — clean dataset!)')

---
## 1. Supervised Learning — Regression

### What is Supervised Learning?

**In plain English:** You hand the algorithm a stack of flash cards. Every card has a question (features X) on one side and the correct answer (label y) on the other. The algorithm studies all the cards until it can answer new questions it has never seen before.

**Technically:** Given a dataset of (X, y) pairs, find a function f such that f(X) ≈ y.

### Regression vs Classification

- **Regression** — the answer is a **number** on a continuous scale  
  → *Predict the exact sale price of a house* (e.g., $487,000)
  
- **Classification** — the answer is a **category** from a fixed set  
  → *Will this house sell ABOVE or BELOW the asking price?* (yes / no)

### Key requirement
You need **labeled data** — someone (or some process) must have recorded the true answer. For house prices that means actual historical sales records. Labeling is expensive, time-consuming, and the main bottleneck in real projects.

In [ ]:
# ── Cell 4: Supervised — Regression ─────────────────────────────────────────
# GOAL: predict the continuous median house VALUE from 8 numerical features

# Split into 80% training data and 20% held-out test data
# random_state ensures the same split every run (reproducibility)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Standardize features: subtract mean, divide by std so all features are on
# a similar scale — important for models that are sensitive to feature magnitude
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_sc  = scaler.transform(X_test)         # only transform test (no fit!)

# Linear Regression: learns weights θ such that ŷ = θ₀ + θ₁x₁ + … + θₙxₙ
reg = LinearRegression()
reg.fit(X_train_sc, y_train)      # this is the "learning" step

# Evaluate on the held-out test set
y_pred_reg = reg.predict(X_test_sc)
mse = mean_squared_error(y_test, y_pred_reg)
r2  = r2_score(y_test, y_pred_reg)

print('=== Supervised Regression: Linear Regression ===')
print(f'Test MSE : {mse:.4f}   (lower = better)')
print(f'Test R²  : {r2:.4f}   (1.0 = perfect, 0 = predicting the mean)')
print(f'\nLearned coefficients (feature importance proxy):')
for feat, coef in zip(X.columns, reg.coef_):
    print(f'  {feat:15s}: {coef:+.4f}')

In [ ]:
# ── Cell 5: Visualise Regression Predictions ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Actual vs Predicted scatter ---
ax = axes[0]
ax.scatter(y_test, y_pred_reg, alpha=0.25, s=8, color='steelblue')
# Perfect prediction line — if the model were perfect all dots would lie here
lims = [y_test.min(), y_test.max()]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Median House Value ($100k)')
ax.set_ylabel('Predicted Value ($100k)')
ax.set_title(f'Regression: Actual vs Predicted\nR² = {r2:.3f}', fontsize=12)
ax.legend(fontsize=9)

# --- Right: Residual distribution ---
residuals = y_test - y_pred_reg   # how far off each prediction is
ax = axes[1]
ax.hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero error')
ax.set_xlabel('Residual (Actual − Predicted, $100k)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution\n(ideal = bell curve centered on 0)', fontsize=12)
ax.legend(fontsize=9)

plt.suptitle('Supervised Learning — Regression Results', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 2. Supervised Learning — Classification

Same paradigm, different output type. Instead of predicting a number, we predict a **category**.

**House price framing:** Create a binary label — is a census block group's median price **above or below** the dataset median? This turns a regression problem into a classification problem, which is a very common real-world transformation (e.g., will this loan default? yes/no).

In [ ]:
# ── Cell 6: Supervised — Classification ──────────────────────────────────────
# Create a binary label: 1 = above median price, 0 = at or below median price
# This simulates a real question: "Will this house sell in the top half of the market?"
median_price = y.median()
y_binary = (y > median_price).astype(int)   # 1 = expensive, 0 = affordable

print(f'Median house value: ${median_price*100:.0f}k')
print(f'Class balance — Affordable (0): {(y_binary==0).sum():,}  |  '
      f'Expensive (1): {(y_binary==1).sum():,}')

# Re-split with the new labels (same random_state → identical train/test indices)
_, _, y_tr_cls, y_te_cls = train_test_split(
    X, y_binary, test_size=0.20, random_state=42
)

# Logistic Regression: outputs a probability P(expensive | features)
# then thresholds at 0.5 → class 0 or 1
clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train_sc, y_tr_cls)   # same scaled X_train from regression cell

y_pred_cls = clf.predict(X_test_sc)
acc = accuracy_score(y_te_cls, y_pred_cls)

print(f'\nTest Accuracy: {acc:.4f}  ({acc*100:.1f}% of houses correctly classified)')
print('\nDetailed report:')
print(classification_report(y_te_cls, y_pred_cls,
                             target_names=['Affordable', 'Expensive']))

---
## 3. Unsupervised Learning

### What is Unsupervised Learning?

**In plain English:** You hand the algorithm a pile of documents with no labels whatsoever. The algorithm must find its own patterns — groups, structures, or compact representations — without being told what to look for.

**Technically:** Given only X (no y), find useful structure in the data.

### Three major sub-types:

| Sub-type | What it does | House price analogy |
|---|---|---|
| **Clustering** | Groups similar data points | Group census blocks into "neighborhood types" |
| **Dimensionality Reduction** | Compresses many features into fewer | Visualize 8 features in 2D |
| **Anomaly Detection** | Finds unusual data points | Flag suspiciously priced listings |
| **Density Estimation** | Models the distribution of the data | Understand the "shape" of the price market |

In [ ]:
# ── Cell 7: Unsupervised — K-Means Clustering ────────────────────────────────
# GOAL: group California census blocks into 5 "neighborhood types"
# without telling the algorithm anything about prices

# Use scaled features so no single feature dominates due to larger magnitude
scaler_full = StandardScaler()
X_scaled = scaler_full.fit_transform(X)   # scale the entire dataset (no test leakage concern — unsupervised)

# KMeans assigns each row to one of n_clusters groups
# It minimises within-cluster sum of squared distances to the cluster center
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)  # returns cluster id (0–4) per row

# Add cluster assignments back to the original DataFrame for analysis
df['cluster'] = cluster_labels

# Summarise each cluster by median house value — did the model separate prices?
cluster_summary = df.groupby('cluster')['MedHouseVal'].agg(['mean', 'median', 'count'])
cluster_summary.columns = ['Mean Value ($100k)', 'Median Value ($100k)', 'Count']
cluster_summary = cluster_summary.sort_values('Median Value ($100k)')

print('K-Means Clustering — 5 Neighborhood Types')
print('(The model never saw house prices, yet clusters differ by price!)')
print(cluster_summary.round(3))

In [ ]:
# ── Cell 8: Unsupervised — PCA Dimensionality Reduction ──────────────────────
# GOAL: compress 8 features → 2 components so we can plot the data in 2D
# PCA finds the directions of maximum variance in the data

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)   # each row is now a 2-element vector

# How much of the total variance does each component capture?
var_ratio = pca.explained_variance_ratio_
print(f'PC1 explains {var_ratio[0]*100:.1f}% of total variance')
print(f'PC2 explains {var_ratio[1]*100:.1f}% of total variance')
print(f'Together   : {var_ratio.sum()*100:.1f}% — information retained after compression')

# Plot: each dot = one census block; colour = K-Means cluster from Cell 7
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: colour by cluster
scatter1 = axes[0].scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=cluster_labels, cmap='tab10', alpha=0.3, s=5
)
axes[0].set_title('PCA 2D — coloured by K-Means Cluster\n(unsupervised groups)', fontsize=11)
axes[0].set_xlabel(f'PC1 ({var_ratio[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({var_ratio[1]*100:.1f}% variance)')
plt.colorbar(scatter1, ax=axes[0], label='Cluster ID')

# Right: colour by actual house price
scatter2 = axes[1].scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=y, cmap='viridis', alpha=0.3, s=5
)
axes[1].set_title('PCA 2D — coloured by Actual House Price\n(the label we never gave the model)', fontsize=11)
axes[1].set_xlabel(f'PC1 ({var_ratio[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({var_ratio[1]*100:.1f}% variance)')
plt.colorbar(scatter2, ax=axes[1], label='Median Value ($100k)')

plt.suptitle('Unsupervised Learning: PCA + KMeans', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 9: Unsupervised — Anomaly Detection ──────────────────────────────────
# GOAL: find census blocks whose feature profiles are unusual
# (might indicate data entry errors, money-laundering, or genuinely odd properties)

# IsolationForest works by randomly isolating points using decision-tree splits.
# Anomalies are easier to isolate (shorter path) than normal points.
# contamination = the fraction of the data we expect to be anomalous
iso = IsolationForest(contamination=0.05, random_state=42)
anomaly_labels = iso.fit_predict(X_scaled)   # +1 = normal, -1 = anomaly

# Count how many were flagged
n_anomalies = (anomaly_labels == -1).sum()
print(f'Isolation Forest flagged {n_anomalies} anomalous census blocks'
      f' ({n_anomalies/len(X)*100:.1f}% of data)')

# Do anomalies have extreme prices?
normal_prices   = y[anomaly_labels ==  1]
anomaly_prices  = y[anomaly_labels == -1]
print(f'Normal  block median price : ${normal_prices.median()*100:.0f}k')
print(f'Anomaly block median price : ${anomaly_prices.median()*100:.0f}k')

# Visualise anomalies in PCA space
fig, ax = plt.subplots(figsize=(8, 6))
colors = np.where(anomaly_labels == 1, 'steelblue', 'red')
ax.scatter(X_pca[:, 0], X_pca[:, 1],
           c=colors, alpha=0.25, s=4)
# Create legend manually
ax.scatter([], [], c='steelblue', label='Normal', s=20, alpha=0.8)
ax.scatter([], [], c='red',       label='Anomaly', s=20, alpha=0.8)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Anomaly Detection (IsolationForest)\nRed = flagged as unusual', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 4. Semi-Supervised Learning

### Plain English First

Imagine you are studying for an exam. Your professor hands back grades on only **10 of your 100 practice problems**. Can you use those 10 graded problems to help yourself do better on the other 90? Yes — by recognising patterns.

That is semi-supervised learning: you have a **small** pool of labeled data and a **large** pool of unlabeled data. The algorithm uses the labels it has to bootstrap learning on the unlabeled examples.

### House Price Context

Suppose you have:
- **500 houses** with professional appraisals (expensive to get — labeled)
- **50,000 houses** with only listing data: sq ft, location, bedrooms, etc. (free — unlabeled)

Semi-supervised approaches use the 500 labeled examples to guide what the model learns from all 50,500 examples.

### Common Algorithms
- **Label Propagation / Label Spreading** (sklearn): spread labels through a similarity graph
- **Self-Training**: train on labeled, predict unlabeled, add high-confidence predictions to labeled set, repeat
- **Generative Models** (VAEs, GANs): learn the data distribution then fine-tune on labels

### Real-World Example
- **Google Photos** face recognition — millions of photos, almost none manually labeled
- **Medical imaging** — expert annotations are costly; semi-supervised leverages the vast unlabeled scan archive

In [ ]:
# ── Cell 10: Semi-Supervised — Self-Training Sketch ───────────────────────────
# We simulate: only 2% of data is labeled; the rest is "unlabeled"
# A simple self-training loop shows the core idea

from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.linear_model import LogisticRegression as LR

# Create a label array where -1 means "unlabeled" (sklearn convention)
LABELED_FRACTION = 0.02       # only 2% of data has labels
y_semisup = y_binary.copy().values  # start with all true labels

# Randomly mask 98% of labels to simulate unlabeled data
unlabeled_mask = np.random.choice(
    len(y_semisup),
    size=int(len(y_semisup) * (1 - LABELED_FRACTION)),
    replace=False
)
y_semisup[unlabeled_mask] = -1    # -1 is sklearn's sentinel for "no label"

labeled_count   = (y_semisup != -1).sum()
unlabeled_count = (y_semisup == -1).sum()
print(f'Labeled  : {labeled_count:,}  ({LABELED_FRACTION*100:.0f}% of data)')
print(f'Unlabeled: {unlabeled_count:,}  ({(1-LABELED_FRACTION)*100:.0f}% of data)')

# SelfTrainingClassifier wraps any base classifier
# It iteratively adds high-confidence unlabeled predictions to the training set
base_clf = LR(max_iter=300, random_state=42)
self_train_clf = SelfTrainingClassifier(
    base_clf,
    threshold=0.85,   # only add predictions with confidence ≥ 85%
    verbose=False
)
self_train_clf.fit(X_scaled, y_semisup)   # fits on all data; uses -1 rows as unlabeled

# Evaluate on test set (the true labels)
y_semisup_pred = self_train_clf.predict(scaler.transform(X_test))
acc_semi = accuracy_score(y_te_cls, y_semisup_pred)

print(f'\nSelf-Training accuracy (2% labels): {acc_semi*100:.1f}%')
print('Compare — Supervised accuracy (100% labels): '
      f'{accuracy_score(y_te_cls, y_pred_cls)*100:.1f}%')
print('\nKey insight: semi-supervised recovers substantial accuracy')
print('without needing expensive labels for every example.')

---
## 5. Self-Supervised Learning

### Plain English First

**Self-supervised learning creates its own labels from the raw data structure — no human ever writes a label.**

Classic example: Given the sentence *"The cat sat on the ___"*, predict the missing word. The rest of the sentence IS the supervision signal. You did not need a human to annotate anything.

### House Price Framing

Take your 50,000 house records. Randomly **mask** one feature (e.g., the number of bedrooms) and train the model to predict it from the remaining features. The model never needs an external label — the masked feature IS the label. After this pre-training, the learned representations transfer well to downstream tasks (like price prediction).

### Why It Matters

- Powers **BERT** (mask words, predict them → language understanding)
- Powers **GPT** (predict the next token → language generation)
- Powers **SimCLR / DINO** (learn visual features from image augmentations)
- Allows learning from **billions** of unlabeled web pages, images, or genomes

In [ ]:
# ── Cell 11: Self-Supervised — Masked Feature Prediction ──────────────────────
# Simulate a simple self-supervised task:
#   INPUT : all 7 features EXCEPT 'AveRooms'
#   TARGET: the masked 'AveRooms' feature
# No external label — the target comes FROM the data itself

MASKED_FEATURE = 'AveRooms'      # the feature we will mask

# Input: all features except the masked one
X_selfsuper = df.drop(columns=['MedHouseVal', 'cluster', MASKED_FEATURE])
# Target: the value of the masked feature — self-generated label
y_selfsuper = df[MASKED_FEATURE]

X_ss_tr, X_ss_te, y_ss_tr, y_ss_te = train_test_split(
    X_selfsuper, y_selfsuper, test_size=0.20, random_state=42
)

# Scale inputs
sc_ss = StandardScaler()
X_ss_tr_sc = sc_ss.fit_transform(X_ss_tr)
X_ss_te_sc = sc_ss.transform(X_ss_te)

# Simple linear model as the "pre-training" network
selfsuper_model = LinearRegression()
selfsuper_model.fit(X_ss_tr_sc, y_ss_tr)

y_ss_pred = selfsuper_model.predict(X_ss_te_sc)
mse_ss = mean_squared_error(y_ss_te, y_ss_pred)
r2_ss  = r2_score(y_ss_te, y_ss_pred)

print(f'=== Self-Supervised: Predict "{MASKED_FEATURE}" from other features ===')
print(f'Test MSE : {mse_ss:.4f}')
print(f'Test R²  : {r2_ss:.4f}  (model learns structure without external labels!)')
print('\nKey concept: in real self-supervised systems, the internal')
print('representations learned during pre-training are REUSED')
print('for downstream tasks (like price prediction) — transfer learning.')

---
## 6. Reinforcement Learning

### Plain English First

A child learning to walk does not read a physics textbook. It tries things, falls, gets feedback (pain / no pain), and gradually learns to keep its balance. That trial-and-error process, guided by rewards, is reinforcement learning.

### Core Components

| Component | Definition | House Market Example |
|---|---|---|
| **Agent** | The learner/decision-maker | An automated property investor |
| **Environment** | The world the agent operates in | The real estate market |
| **State** | The current situation | Current prices, interest rates, inventory |
| **Action** | What the agent can do | Buy / Sell / Hold a property |
| **Reward** | Feedback signal | Profit or loss from each decision |
| **Policy** | The strategy the agent learns | When to buy, sell, or hold |

### The RL Loop

```
Agent observes State → Takes Action → Gets Reward → Updates Policy → repeat
```

### Key Algorithms
- **Q-Learning** / **DQN** — learn the value of each (state, action) pair  
- **Policy Gradient (REINFORCE)** — directly optimise the policy  
- **PPO (Proximal Policy Optimisation)** — used in ChatGPT's RLHF training

### Why RL Is Different
- No fixed dataset — the agent **generates its own experience**
- Rewards may be **delayed** (sell price is only known months later)
- Exploration vs. exploitation trade-off: try new strategies vs. stick to what works

> **RL is NOT the right tool for most tabular prediction tasks.**  
> Use it when decisions are sequential and actions affect future states.

In [ ]:
# ── Cell 12: RL Concept — Simple Q-Table Toy Example ─────────────────────────
# We simulate a TINY property market with 3 states and 3 actions
# This demonstrates the Q-learning update rule, not a real RL system

STATES  = ['Market Dipping', 'Market Stable', 'Market Rising']
ACTIONS = ['Buy', 'Hold', 'Sell']

# Hypothetical reward table: reward[state][action]
# Positive = profit, negative = loss (in $10k units)
reward_table = {
    'Market Dipping': {'Buy': +5,  'Hold': -1, 'Sell': -8},
    'Market Stable' : {'Buy':  0,  'Hold':  1, 'Sell':  2},
    'Market Rising' : {'Buy': -3,  'Hold':  4, 'Sell': +9},
}

# Q-table: stores the LEARNED value of each (state, action) pair
# Initialised to zero — the agent knows nothing at the start
q_table = {s: {a: 0.0 for a in ACTIONS} for s in STATES}

# Q-learning hyperparameters
alpha   = 0.5    # learning rate: how fast to update Q values
gamma   = 0.9    # discount factor: how much to value future rewards

# Simulate 200 random (state, action) experiences and update Q
np.random.seed(42)
for _ in range(200):
    state  = np.random.choice(STATES)
    action = np.random.choice(ACTIONS)
    reward = reward_table[state][action]
    next_state = np.random.choice(STATES)
    
    # Q-learning update rule:  Q(s,a) ← Q(s,a) + α[r + γ·max Q(s') − Q(s,a)]
    best_next_q = max(q_table[next_state].values())  # greedy future value
    q_table[state][action] += alpha * (
        reward + gamma * best_next_q - q_table[state][action]
    )

# Display the learned Q-table
q_df = pd.DataFrame(q_table).T
q_df.index.name = 'State'
print('Learned Q-Table (value of each state-action pair):')
print(q_df.round(2))
print('\nBest action per state (greedy policy):')
for state in STATES:
    best_action = max(q_table[state], key=q_table[state].get)
    print(f'  {state:18s} → {best_action}')

---
## 7. Visual Taxonomy Diagram

In [ ]:
# ── Cell 13: ML Taxonomy — Matplotlib Box Diagram ─────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16); ax.set_ylim(0, 9)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

def draw_box(ax, x, y, w, h, label, sublabel='', color='#4C72B0', fontsize=11):
    """Helper: draw a filled rectangle with centred text."""
    rect = mpatches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle='round,pad=0.1',
        facecolor=color, edgecolor='white', linewidth=2, alpha=0.9
    )
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.15 if sublabel else 0),
            label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white')
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.25, sublabel,
                ha='center', va='center', fontsize=8, color='#e0e0e0')

def draw_arrow(ax, x1, y1, x2, y2):
    """Helper: draw an annotated arrow between two points."""
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Root: Machine Learning
draw_box(ax, 5.5, 7.4, 5, 1.2, 'Machine Learning', color='#2d2d2d', fontsize=14)

# Main branches
branches = [
    (0.2,  4.8, 3.0, 1.2, 'Supervised',      'Has labels y',       '#1f77b4'),
    (3.5,  4.8, 3.0, 1.2, 'Unsupervised',    'No labels',          '#2ca02c'),
    (7.0,  4.8, 3.0, 1.2, 'Semi-Supervised', 'Few labels',         '#ff7f0e'),
    (10.5, 4.8, 3.0, 1.2, 'Self-Supervised', 'Self-generated lbls','#9467bd'),
    (13.0, 4.8, 2.8, 1.2, 'Reinforcement',   'Reward signals',     '#d62728'),
]
for (bx, by, bw, bh, bl, bsl, bc) in branches:
    draw_box(ax, bx, by, bw, bh, bl, bsl, color=bc, fontsize=10)
    draw_arrow(ax, 8.0, 7.4, bx + bw/2, by + bh)   # from root to branch top

# Sub-types
subtypes = [
    # Supervised
    (0.2,  2.8, 1.35, 1.0, 'Regression',      '#5b9bd5'),
    (1.65, 2.8, 1.35, 1.0, 'Classification',  '#5b9bd5'),
    # Unsupervised
    (3.5,  2.8, 1.35, 1.0, 'Clustering',      '#57a057'),
    (4.95, 2.8, 1.55, 1.0, 'Dim. Reduction',  '#57a057'),
    # Semi-supervised
    (7.0,  2.8, 1.35, 1.0, 'Label Prop.',     '#ffaa55'),
    (8.45, 2.8, 1.45, 1.0, 'Self-Training',   '#ffaa55'),
    # Self-supervised
    (10.5, 2.8, 1.35, 1.0, 'Masked LM',       '#b47cc7'),
    (11.95,2.8, 1.45, 1.0, 'Contrastive',     '#b47cc7'),
    # RL
    (13.0, 2.8, 1.25, 1.0, 'Q-Learning',      '#e05757'),
    (14.35,2.8, 1.35, 1.0, 'Policy Grad.',    '#e05757'),
]
for (sx, sy, sw, sh, sl, sc) in subtypes:
    draw_box(ax, sx, sy, sw, sh, sl, color=sc, fontsize=8)

# Arrows from branches to sub-types
arrow_pairs = [
    (1.70, 4.8, 0.88, 3.8), (1.70, 4.8, 2.32, 3.8),  # Supervised
    (5.00, 4.8, 4.17, 3.8), (5.00, 4.8, 5.72, 3.8),  # Unsupervised
    (8.50, 4.8, 7.67, 3.8), (8.50, 4.8, 9.17, 3.8),  # Semi-sup
    (12.0, 4.8, 11.17,3.8), (12.0, 4.8, 12.67,3.8),  # Self-sup
    (14.4, 4.8, 13.62,3.8), (14.4, 4.8, 15.02,3.8),  # RL
]
for (ax1, ay1, ax2, ay2) in arrow_pairs:
    draw_arrow(ax, ax1, ay1, ax2, ay2)

# House price examples at bottom
examples = [
    (0.2,  0.8, 3.0, 1.5, 'Predict exact\nsale price\n(Linear Reg.)',    '#add8e6'),
    (3.5,  0.8, 3.0, 1.5, 'Group census\nblocks into\nneighbourhoods',    '#90EE90'),
    (7.0,  0.8, 3.0, 1.5, '500 appraisals\n+ 50k listings\n→ price model','#FFD580'),
    (10.5, 0.8, 3.0, 1.5, 'Predict masked\nbedroom count\nfrom other feat','#DDA0DD'),
    (13.0, 0.8, 2.8, 1.5, 'Buy/Sell/Hold\nproperty based\non market state','#F08080'),
]
for (ex, ey, ew, eh, el, ec) in examples:
    rect = mpatches.FancyBboxPatch((ex, ey), ew, eh,
        boxstyle='round,pad=0.1', facecolor=ec, edgecolor='#aaa', linewidth=1, alpha=0.85)
    ax.add_patch(rect)
    ax.text(ex+ew/2, ey+eh/2, el, ha='center', va='center', fontsize=8, color='#333')

ax.text(8, 0.45, '🏠 House Price Application', ha='center', va='center',
        fontsize=10, color='#555', style='italic')

ax.set_title('Machine Learning Taxonomy', fontsize=18, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

---
## 8. Comparison Table

| Paradigm | Needs Labels? | Example Algorithm | House Price Use Case | Data Volume Needed |
|---|---|---|---|---|
| Supervised | Yes (all data) | LinearRegression, SVM | Predict exact sale price | Moderate (thousands) |
| Unsupervised | No | KMeans, PCA | Group neighbourhoods | Low–Large |
| Semi-Supervised | Yes (small fraction) | SelfTrainingClassifier | 500 appraisals + 50k listings | Large unlabeled |
| Self-Supervised | No (self-generated) | Masked Autoencoder, BERT | Predict masked bedroom count | Very large |
| Reinforcement | No (rewards) | Q-Learning, PPO | Buy/Sell/Hold decisions | Simulated environment |

---

## 9. Decision Flowchart

```
START: What kind of ML problem do I have?
│
├─► Do I need to make SEQUENTIAL DECISIONS over time?
│     YES → REINFORCEMENT LEARNING (RL)
│
├─► Do I have LABELED training examples (y values)?
│     │
│     ├─► YES, ALL examples labeled
│     │     ├─► Output is continuous (number) → REGRESSION (supervised)
│     │     └─► Output is a category        → CLASSIFICATION (supervised)
│     │
│     └─► ONLY SOME examples labeled
│           ├─► Labels created from data structure itself → SELF-SUPERVISED
│           └─► Human-created labels, most missing       → SEMI-SUPERVISED
│
└─► NO LABELS AT ALL — find structure in X only
       ├─► Group similar points  → CLUSTERING
       ├─► Compress features     → DIMENSIONALITY REDUCTION
       └─► Find unusual points   → ANOMALY DETECTION
```

In [ ]:
# ── Cell 14: Elbow Method — How Many Clusters? ───────────────────────────────
# When using K-Means you need to choose k (number of clusters) ahead of time.
# The "elbow method" plots inertia (within-cluster sum of squares) vs k.
# The "elbow" — where inertia stops dropping steeply — suggests the best k.

inertias = []
k_range = range(1, 11)   # try k = 1 through 10

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=5)
    km.fit(X_scaled)
    inertias.append(km.inertia_)   # inertia = sum of squared distances to centroid

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_range, inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axvline(5, color='red', linestyle='--', alpha=0.6, label='k=5 (our choice)')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (within-cluster SS)')
ax.set_title('Elbow Method — Choosing k for K-Means\n'
             '(look for where the curve bends sharply)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print('Observation: inertia drops steeply from k=1 → k=3–5,')
print('then levels off. k=5 is a reasonable elbow point.')

In [ ]:
# ── Cell 15: Full Summary Visualisation ──────────────────────────────────────
# One figure showing all five paradigms side-by-side using our results

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: Supervised Regression (actual vs predicted) ---
ax = axes[0]
ax.scatter(y_test[:500], y_pred_reg[:500], alpha=0.3, s=10, color='#1f77b4')
lim = [y_test.min(), y_test.max()]
ax.plot(lim, lim, 'r--', lw=1.5)
ax.set_title('Supervised (Regression)\nActual vs Predicted Price', fontsize=11)
ax.set_xlabel('Actual ($100k)'); ax.set_ylabel('Predicted ($100k)')

# --- Panel 2: Unsupervised Clustering in PCA space ---
ax = axes[1]
scatter = ax.scatter(
    X_pca[:2000, 0], X_pca[:2000, 1],
    c=cluster_labels[:2000], cmap='tab10', alpha=0.4, s=10
)
ax.set_title('Unsupervised (Clustering)\n5 Neighbourhood Groups (PCA view)', fontsize=11)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.colorbar(scatter, ax=ax, label='Cluster')

# --- Panel 3: Anomaly detection ---
ax = axes[2]
colors3 = ['red' if a == -1 else 'steelblue' for a in anomaly_labels[:2000]]
ax.scatter(X_pca[:2000, 0], X_pca[:2000, 1],
           c=colors3, alpha=0.3, s=10)
ax.scatter([], [], c='red',       label='Anomaly', s=30)
ax.scatter([], [], c='steelblue', label='Normal',  s=30)
ax.set_title('Unsupervised (Anomaly Detection)\nIsolation Forest Flags', fontsize=11)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(fontsize=9)

plt.suptitle('ML Paradigm Showcase — California Housing Dataset',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Self-Check Questions

Test your understanding before moving on. Try to answer without looking at the answers.

---

### Question 1
You want to predict whether a neighbourhood's median price will **rise or fall** next year. You have historical price data but **no "rise/fall" labels**. What type of ML problem is this, and what approach would you take?

<details>
<summary>Click to reveal answer</summary>

**Answer:**  
This is a **supervised classification** problem in structure — the output is a binary label (rise/fall) — but you do not have the labels yet.  

Your first step is **label engineering**: compute year-over-year price changes from historical data and threshold them into rise (1) / fall (0). Once you have constructed those labels, it becomes a standard supervised binary classification problem.  

Algorithms to consider: Logistic Regression (baseline), Random Forest, Gradient Boosting (XGBoost/LightGBM), or a neural network.

If constructing labels is impossible (e.g., truly no signal), you could explore **unsupervised time-series clustering** to find neighbourhoods with similar price trajectories, then manually inspect clusters.

</details>

---

### Question 2
You have 1,000 labeled house appraisals and 100,000 houses with only listing data (no price labels). Which ML paradigm fits best, and why?

<details>
<summary>Click to reveal answer</summary>

**Answer: Semi-Supervised Learning**  

- You have a **small** amount of labeled data (1,000 appraisals) and a **large** amount of unlabeled data (100,000 listings) — this is the defining signature of semi-supervised problems.
- A purely supervised model trained on only 1,000 examples will likely underfit. Pure unsupervised ignores the valuable labels entirely.
- **Practical approach:** Use `SelfTrainingClassifier` or `LabelSpreading` (sklearn), or train an autoencoder on all 101,000 examples to learn rich representations, then fine-tune on the 1,000 labeled examples.

</details>

---

### Question 3
What is the key difference between **self-supervised** and **semi-supervised** learning?

<details>
<summary>Click to reveal answer</summary>

**Answer:**  

| Aspect | Semi-Supervised | Self-Supervised |
|---|---|---|
| **Where labels come from** | Humans label a small subset | Labels are **generated automatically** from the data structure |
| **Requires human labeling?** | Yes (for the small labeled portion) | No — zero human annotation |
| **Scalability** | Limited by labeling budget | Scales to billions of examples |
| **Example** | 500 human appraisals + 50k unlabeled | Predict masked bedroom count from other features |

In self-supervised learning, the task itself (e.g., predict the masked word, predict the next token) is synthetic and derived from the raw data. This is what makes BERT, GPT, and modern vision models so powerful — they exploit massive unlabeled corpora.

</details>

---

### Question 4 (Bonus)
KMeans on the California Housing data found that different clusters have different median prices — even though the model **never saw prices** during training. Why does this happen?

<details>
<summary>Click to reveal answer</summary>

**Answer:**  
The features in the dataset (median income `MedInc`, average rooms `AveRooms`, latitude/longitude, etc.) are **correlated with house prices**. Higher-income, larger-room, coastal neighbourhoods tend to cost more. KMeans groups houses by feature similarity, so naturally, groups that share high income, large rooms, and coastal geography also share high prices — even though price was never used as a feature.

This illustrates a key insight: **unsupervised structure often captures meaningful signal**, and the groups can be used as features for downstream supervised tasks (a technique called *cluster-then-predict* or *representation learning*).

</details>

---

### Question 5 (Bonus)
A friend says "Just use deep learning for everything — it works for supervised, unsupervised, AND reinforcement learning!" Critique this statement.

<details>
<summary>Click to reveal answer</summary>

**Answer:**  
Your friend is technically correct — deep learning architectures can be adapted to all paradigms. But the statement ignores important considerations:

1. **Data requirements**: Deep learning typically needs large datasets. On 1,000 labeled examples, a regularised linear model or gradient-boosted trees often outperforms a neural network.
2. **Interpretability**: Linear models and decision trees are explainable; neural networks are not. In regulated industries (finance, healthcare), explainability may be legally required.
3. **Compute cost**: Training large networks requires GPUs and significant engineering. A logistic regression trains in seconds on a laptop.
4. **Overfitting risk**: Without sufficient data, deep models overfit dramatically.

**Rule of thumb**: start with the simplest model that could work, then escalate complexity only if simpler models fall short on your metric.

</details>